# 투자 데이터 출처 선택 실습

이 노트북을 둔 작업 폴더에서 Python 3 커널을 시작하고 위에서 아래로 실행합니다. 패키지 설치는 필요하지 않습니다. API 키·계정·비밀값을 읽는 코드는 없습니다.

기본 모드는 **조회**입니다. 실행 결과는 `outputs/실행시각_UUID/`에 새로 저장하며 기존 입력을 덮어쓰지 않습니다. 네트워크 실패를 가상 자료나 저장본으로 자동 대체하지 않습니다. 이 배포본의 코드는 실행하지 않았으므로 관측값은 실행 후 확인합니다.


## 1. 준비

`MODE`는 `"조회"` 또는 `"저장본"`으로 선택합니다. 표준 라이브러리는 Python 버전을, 외부 패키지는 설치된 배포판 버전을 출력합니다. 조회 셀을 다시 실행해도 새 실행 폴더를 만듭니다.


In [ ]:
from pathlib import Path
import pandas as pd
import FinanceDataReader as fdr
import json
import hashlib
import importlib.metadata
import requests
import sys
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import display

MODE = "조회"
BASE = Path.cwd().resolve()
VERSIONS = {
    "Python": sys.version.split()[0],
    "pandas": importlib.metadata.version("pandas"),
    "FinanceDataReader": importlib.metadata.version("finance-datareader"),
    "requests": importlib.metadata.version("requests"),
}
print("실행 모드:", MODE)
print("버전 정보:")
for name, version in VERSIONS.items():
    print(f"{name}: {version}")
print("pathlib, json, hashlib, importlib.metadata: Python 표준 라이브러리")


def local_path(relative):
    # 심볼릭 링크를 통해 작업 폴더 밖으로 나가는 것도 차단합니다.
    path = (BASE / relative).resolve()
    if not path.is_relative_to(BASE):
        raise ValueError("작업 폴더 안의 경로만 사용할 수 있습니다.")
    return path


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, value):
    # 기존 파일이 있으면 덮어쓰지 않고 실패합니다.
    with path.open("x", encoding="utf-8") as file:
        json.dump(value, file, ensure_ascii=False, indent=2)


def normalize_prices(frame):
    frame = frame.copy()
    required = {"Date", "Close", "Volume"}
    if not required.issubset(frame.columns):
        raise ValueError("필수 열이 없습니다.")
    frame["Date"] = pd.to_datetime(frame["Date"], errors="raise")
    if frame["Date"].isna().any():
        raise ValueError("날짜가 비어 있습니다.")
    # 정렬·중복 제거·열 이름 변경 없이 원래 표를 관찰합니다.
    return frame


## 2. 이번 가격의 출처

- 대상: **KODEX 200**, 종목코드 `069500`
- 요청 기간: **2024-01-02~2024-01-17**
- 수집 도구: **FinanceDataReader**, 직접 제공처: **네이버 금융**, 거래 시장: **KRX**
- 명시적 조회 경로: `NAVER:069500`

이번 실습은 공식 KRX OPEN API를 직접 호출하지 않습니다. 네이버에서 받은 `Close`와 공식 원본의 같은 날짜·같은 가격 정의를 대조하기 전에는 공식 검증 완료로 표시하지 않습니다.


In [ ]:
REQUEST = {
    "name": "KODEX 200", "symbol": "069500", "reader_symbol": "NAVER:069500",
    "start": "2024-01-02", "end": "2024-01-17",
}
PROVIDER = "네이버 금융"
MARKET = "KRX"
PENDING = "보류 — 같은 날짜·가격 정의의 공식 원본 미확보"


## 3~4. 조회 또는 저장본 읽기

조회는 `requests.sessions.Session.request`에 연결 10초·읽기 30초 제한을 적용합니다. `finally`에서 원래 함수를 복원합니다. 이 제한은 개별 요청의 연결·읽기 제한이며 전체 작업의 총시간 제한은 아닙니다. 이 셀과 동시에 다른 네트워크 셀을 실행하지 마세요.

저장본 모드는 사용자가 이미 확보한 `data/price-snapshot.csv`와 `data/snapshot-metadata.json`을 사용합니다. 저장본은 자동 생성하지 않습니다. 성공한 조회의 `prices.csv`와 `metadata.json`을 각각 이 이름으로 **기존 입력이 없는 경우에만** 복사해 준비할 수 있습니다. 메타데이터의 `sha256`은 CSV 원본 바이트의 해시이며 `collected_at`, `request`, `provider`, `market`, `collector_version`을 함께 보존해야 합니다. 해시는 파일 일치 확인이며 공식 가격의 정확성을 보증하지 않습니다.

두 모드 모두 같은 날짜 해석 함수를 사용합니다. 실패하면 실제 예외 클래스 이름과 경로를 포함하지 않는 짧은 안내를 출력합니다. 원래 예외 문자열은 개인 절대 경로가 섞일 수 있어 저장하거나 출력하지 않습니다.


In [ ]:
# 이 셀을 실행할 때마다 이전 결과와 구분되는 폴더를 만듭니다.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex
run_dir = local_path(Path("outputs") / run_id)
run_dir.mkdir(parents=True, exist_ok=False)
prices = None
metadata = None
failure = None
started_at = now_iso()
stage = "모드 확인"

try:
    if MODE not in ("조회", "저장본"):
        raise ValueError("지원하지 않는 실행 모드입니다.")

    if MODE == "조회":
        stage = "네이버 가격 조회"
        original_request = requests.sessions.Session.request

        def request_with_timeout(self, method, url, **kwargs):
            kwargs["timeout"] = (10, 30)
            return original_request(self, method, url, **kwargs)

        try:
            requests.sessions.Session.request = request_with_timeout
            candidate = fdr.DataReader("NAVER:069500", "2024-01-02", "2024-01-17").reset_index()
        finally:
            requests.sessions.Session.request = original_request

        collected_at = now_iso()
        source_version = VERSIONS["FinanceDataReader"]
        snapshot_info = None
    else:
        stage = "저장본 파일 확인"
        snapshot_csv = local_path("data/price-snapshot.csv")
        snapshot_json = local_path("data/snapshot-metadata.json")
        saved = json.loads(snapshot_json.read_text(encoding="utf-8"))
        csv_bytes = snapshot_csv.read_bytes()
        actual_hash = hashlib.sha256(csv_bytes).hexdigest()
        stage = "저장본 해시 확인"
        if actual_hash != saved["sha256"]:
            raise ValueError("저장본 해시가 일치하지 않습니다.")
        stage = "저장본 출처 확인"
        if saved["request"] != REQUEST or saved["provider"] != PROVIDER or saved["market"] != MARKET:
            raise ValueError("저장본의 요청 또는 출처가 다릅니다.")
        collected_at = saved["collected_at"]
        if not isinstance(collected_at, str) or not collected_at.strip():
            raise ValueError("저장본 수집시각이 없습니다.")
        datetime.fromisoformat(collected_at.replace("Z", "+00:00"))
        source_version = saved["collector_version"]
        # 해시 검사한 바로 그 바이트를 파싱합니다.
        from io import BytesIO
        candidate = pd.read_csv(BytesIO(csv_bytes))
        snapshot_info = {
            "csv": "data/price-snapshot.csv",
            "metadata": "data/snapshot-metadata.json", "sha256": actual_hash,
        }
        print("저장본 원래 수집시각:", collected_at)
        print("파일 해시 일치. 이번 실행은 재조회가 아닙니다.")

    stage = "수집 행 수 확인"
    if len(candidate) == 0:
        raise ValueError("수집 결과가 0행입니다.")
    stage = "날짜와 필수 열 해석"
    candidate = normalize_prices(candidate)
    stage = "결과 저장"
    csv_path = run_dir / "prices.csv"
    candidate.to_csv(csv_path, index=False, encoding="utf-8", date_format="%Y-%m-%d", mode="x")
    metadata = {
        "status": "성공", "mode": MODE, "request": REQUEST,
        "started_at": started_at, "collected_at": collected_at, "processed_at": now_iso(),
        "actual_period": {
            "start": candidate["Date"].min().strftime("%Y-%m-%d"),
            "end": candidate["Date"].max().strftime("%Y-%m-%d"),
        },
        "provider": PROVIDER, "market": MARKET, "collector": "FinanceDataReader",
        "collector_version": source_version, "versions": VERSIONS,
        "rows": len(candidate), "columns": list(candidate.columns),
        "price_file": "prices.csv", "sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
        "snapshot_input": snapshot_info, "official_verification": PENDING,
    }
    write_json(run_dir / "metadata.json", metadata)
    prices = candidate
    print("가격 조회 성공" if MODE == "조회" else "저장본 읽기 성공")
    print("공식 가격 대조:", PENDING)
except Exception as exc:
    # 원문 대신 단계별 메시지를 사용하여 개인 절대 경로를 노출하지 않습니다.
    messages = {
        "모드 확인": "실행 모드는 조회 또는 저장본이어야 합니다.",
        "네이버 가격 조회": "네이버 가격 요청에 실패했습니다. 연결 또는 응답을 확인하세요.",
        "저장본 파일 확인": "저장본 두 파일의 존재와 JSON 형식을 확인하세요.",
        "저장본 해시 확인": "저장본 SHA-256이 없거나 일치하지 않습니다.",
        "저장본 출처 확인": "저장본의 요청·출처·수집시각·버전 정보를 확인하세요.",
        "수집 행 수 확인": "수집 결과가 0행이므로 실패로 기록했습니다.",
        "날짜와 필수 열 해석": "Date·Close·Volume 열과 날짜 형식을 확인하세요.",
        "결과 저장": "실행 폴더에 결과를 저장하지 못했습니다.",
    }
    failure = {
        "status": "실패", "mode": MODE, "request": REQUEST, "failed_at": now_iso(),
        "stage": stage, "exception_type": type(exc).__name__, "message": messages[stage],
        "provider": PROVIDER, "versions": VERSIONS,
    }
    write_json(run_dir / "failure.json", failure)
    print(f"{failure['exception_type']}: {failure['message']}")
    print("자동 대체 없이 실패 상태를 유지합니다.")

print("이번 실행 폴더:", run_dir.relative_to(BASE).as_posix())


## 5. 표 확인: 실습 기대값과 관측값

고정된 이 기간의 실습 기대값은 **12행·7열**입니다. 실제 행 수는 관찰값이며 거래일 완전성 보증이 아닙니다. 기대값과 달라도 행을 만들어 채우거나 지우지 않습니다. 거래일 완전성은 별도로 공식 거래일 달력과 대조해야 합니다.


In [ ]:
observations = None
if prices is None:
    print("가격 자료가 없어 표 확인을 수행하지 않았습니다.")
else:
    observations = {
        "전체 행 수": len(prices), "전체 열 수": len(prices.columns),
        "첫날": prices["Date"].min().strftime("%Y-%m-%d"),
        "마지막날": prices["Date"].max().strftime("%Y-%m-%d"),
        "열 이름": list(prices.columns),
        "열별 빈 값 수": {name: int(value) for name, value in prices.isna().sum().items()},
        "중복 날짜 수": int(prices["Date"].duplicated().sum()),
        "날짜 오름차순 정렬 여부": bool(prices["Date"].is_monotonic_increasing),
        "기대값 12행·7열 일치": prices.shape == (12, 7),
    }
    print("표 확인 결과")
    for label, value in observations.items():
        print(f"{label}: {value}")
    print("행 수는 관찰값이며 거래일 완전성 보증이 아닙니다.")
    print("날짜·종가·거래량 전체 표")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(prices.loc[:, ["Date", "Close", "Volume"]])


## 6. 값 하나 확인

2024-01-02의 `Close`를 확인합니다. **값을 조회한 성공과 공식 대조 성공은 다릅니다.** 같은 날짜·가격 정의의 공식 원본을 확보하지 않았으므로 대조는 보류합니다. 최신 가격이나 NAV로 대신 비교하지 않습니다.


In [ ]:
close_text = "확인하지 못함"
if prices is None:
    print("가격 자료가 없어 값을 확인하지 못했습니다.")
else:
    selected = prices.loc[prices["Date"].eq(pd.Timestamp("2024-01-02")), ["Date", "Close"]]
    print("2024-01-02 종가 확인 표")
    display(selected)
    if len(selected) == 1 and selected["Close"].notna().all():
        close_text = str(selected["Close"].iloc[0])
        print("2024-01-02 Close:", close_text)
    else:
        close_text = "해당 날짜가 없거나 중복되었거나 Close가 비어 있어 단일 값 확인 보류"
        print(close_text)
print("공식 가격 대조:", PENDING)


## 7. 출처 선택표 저장

공식 안내: [KRX 이용방법](https://openapi.krx.co.kr/contents/OPP/INFO/OPPINFO003.jsp), [OpenDART 재무정보](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS003), [한국은행 ECOS](https://ecos.bok.or.kr/api/).

무료 조회와 데이터 공개 재배포 허가는 다릅니다. 재배포 조건은 **미확인**입니다. `verified_on`은 공식 데이터 대조 완료일이며 이번에는 빈 값으로 둡니다. 가격 수집 도구 버전은 `collector_version` 추가 열에 기록합니다. KRX·DART·ECOS API는 이번에 호출하지 않습니다.


In [ ]:
naver_status = (
    "조회 성공" if MODE == "조회" else "저장본 해시 확인 후 읽기 성공; 재조회 아님"
) if prices is not None else "실패; 자동 대체 없음"
reuse_note = "재배포 조건 미확인; 무료 조회와 공개 재배포 허가는 별개"
source_rows = [
    {
        "data_kind": "price_naver", "source_owner": "네이버 금융; 거래 시장 KRX",
        "access_path": "NAVER:069500; https://finance.naver.com/item/main.naver?code=069500",
        "collector": "FinanceDataReader", "access_status": naver_status,
        "verification_status": PENDING, "reuse_note": reuse_note, "verified_on": "",
        "collector_version": metadata["collector_version"] if prices is not None else VERSIONS["FinanceDataReader"],
    },
    {
        "data_kind": "price_krx", "source_owner": "한국거래소 KRX",
        "access_path": "https://openapi.krx.co.kr/contents/OPP/INFO/OPPINFO003.jsp",
        "collector": "미사용", "access_status": "회원·인증키·서비스 승인 필요; 이번 미호출",
        "verification_status": PENDING, "reuse_note": reuse_note, "verified_on": "",
        "collector_version": "해당 없음; 미호출",
    },
    {
        "data_kind": "financials_dart", "source_owner": "금융감독원 OpenDART; 기업이 제출한 재무제표",
        "access_path": "https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS003",
        "collector": "미사용", "access_status": "인증키 필요; 이번 미호출",
        "verification_status": "미검증 — 재무제표 미수집", "reuse_note": reuse_note, "verified_on": "",
        "collector_version": "해당 없음",
    },
    {
        "data_kind": "rates_ecos", "source_owner": "한국은행 ECOS",
        "access_path": "https://ecos.bok.or.kr/api/",
        "collector": "미사용", "access_status": "통계표·항목·주기·단위를 선택; 인증키 확인 후 사용; 이번 미호출",
        "verification_status": "미검증 — 통계 미수집", "reuse_note": reuse_note, "verified_on": "",
        "collector_version": "해당 없음",
    },
]
sources = pd.DataFrame(source_rows)
sources.to_csv(run_dir / "sources.csv", index=False, encoding="utf-8", mode="x")
print("출처 선택표")
display(sources)


## 8. 이번 실행 기록 저장

관측값과 실패 여부를 구분하여 같은 실행 폴더의 `data-source-selection.md`에 기록합니다. 결과 저장 셀만 다시 실행하면 덮어쓰기 방지를 위해 실패할 수 있습니다. 새 실행은 3~4절부터 순서대로 진행합니다.


In [ ]:
files = sorted(path.name for path in run_dir.iterdir() if path.is_file())
files.append("data-source-selection.md")
observation_text = (
    "\n".join(f"- {label}: {value}" for label, value in observations.items())
    if observations is not None else "- 관측값 없음: 조회 또는 저장본 읽기 실패"
)
status_text = naver_status
failure_text = (
    f"{failure['exception_type']}: {failure['message']}"
    if failure is not None else "없음"
)
collection_text = metadata["collected_at"] if prices is not None else "수집 성공 시각 없음"
report = f"""# 투자 데이터 출처 선택 기록

- 실행 모드: {MODE}
- 실행 폴더: {run_dir.relative_to(BASE).as_posix()}
- 이번 파일: {', '.join(files)}
- 처리 상태: {status_text}
- 실패 내용: {failure_text}
- 원래 수집시각: {collection_text}
- 저장본 여부: {'저장본 사용; 재조회 아님' if MODE == '저장본' else '네이버 조회 모드'}
- 요청: KODEX 200(069500), 2024-01-02~2024-01-17

## 관측값
{observation_text}
- 2024-01-02 Close: {close_text}
- 고정 실습 기대값: 12행·7열
- 행 수는 관찰값이며 거래일 완전성 보증이 아닙니다.

## 출처와 보류 사유
수집 도구는 FinanceDataReader, 직접 제공처는 네이버 금융, 거래 시장은 KRX입니다.
공식 KRX OPEN API를 직접 호출하지 않았습니다. 도구 버전과 접근 상태는 sources.csv에 기록합니다.
공식 가격 대조 상태: {PENDING}
값 조회 성공과 공식 대조 성공은 별개입니다. 최신 가격이나 NAV로 대신 비교하지 않습니다.
무료 조회와 공개 재배포 허가는 다르며 재배포 조건은 미확인입니다.

## 다음 확인
- KRX의 회원·인증키·서비스 승인 절차를 확인하고 같은 날짜의 공식 가격 원본을 확보합니다.
- 종목·날짜·종가 정의·수정주가 여부·단위를 맞춘 뒤 2024-01-02 Close를 대조합니다.
- 공식 거래일 달력과 날짜 집합을 비교하여 거래일 완전성을 별도로 확인합니다.
- OpenDART는 기업 제출 재무제표의 보고서·연결/별도·정정 여부를 선택하고 인증키 요건을 확인합니다.
- ECOS는 통계표·항목·주기·단위를 선택하고 인증키 요건을 확인합니다.
- 공개 재배포 전 각 제공처의 이용·재배포 조건을 확인합니다.

공식 안내: [KRX](https://openapi.krx.co.kr/contents/OPP/INFO/OPPINFO003.jsp),
[OpenDART](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS003),
[ECOS](https://ecos.bok.or.kr/api/).
"""
with (run_dir / "data-source-selection.md").open("x", encoding="utf-8") as file:
    file.write(report)
print("실행 기록 저장:", (run_dir / "data-source-selection.md").relative_to(BASE).as_posix())


## 9. 의도적 오류 실험

**파일명 오류를 의도적으로 만든 실험**입니다. 올바른 `data/price-snapshot.csv`에서 글자 하나를 뺀 `data/price-snapsho.csv`를 읽습니다. `FileNotFoundError`를 잡아 오류 이름과 잘못된 상대 파일명만 출력합니다. 앞선 조회 결과와 저장 파일은 보존합니다. 이 마지막 셀은 이후 수정 요청에 사용합니다.


In [ ]:
wrong_relative = "data/price-snapsho.csv"
print("파일명 오류를 의도적으로 만든 실험")
try:
    pd.read_csv(local_path(wrong_relative))
except FileNotFoundError as exc:
    print(f"{type(exc).__name__}: {wrong_relative}")
    print("앞선 조회 결과와 저장 파일은 보존되었습니다.")
else:
    print("해당 상대 파일이 이미 있어 의도한 파일 없음 오류가 발생하지 않았습니다.")
